# Gene-set analyses of the comorbidity sub-graphs

The gene-set counterpart of `4_10`: instead of drawing the sub-graph around each interface protein, we turn it into a **gene set** and confront it with the bulk RNA-seq DE gene lists of `4_00`.

Two analyses, on two pairings.

**The analyses.** *GOAT* tests the sub-graph's gene set for enrichment against the whole effect-size ranking of a gene list (the R `goat` package, via `rpy2`). The *intersection* is its naive counterpart — how much of the gene set is simply among the genes the list marks significant. They want different namespaces: GOAT matches the gene lists' `gene` column, which is the HGNC dataset's `entrez_id`, so its gene sets are `ncbigene`; the intersection matches the `symbol` column, so its gene sets are `hgnc.symbol`.

**The pairings.** COVID → PD, whose downstream side is the PD activity-flow maps, and COVID → AD, whose downstream side is the **AD BEL knowledge graph** — the same two `4_10` draws, with the same collection names, walk and `max_levels`. The AD side used to be out of reach here: BEL nodes carried only `uniprot` annotations, and they were labelled `BELModelElement` but not `ModelElement`, which is what the annotation helpers in `commute_dm.queries` match — so they returned nothing for a BEL node, silently. `2_00` now writes `hgnc.symbol`, `ncbigene`, `hgnc` and `uniprot` on the KG's HGNC-encoded proteins and labels its nodes `ModelElement`, which is all it took.

Nothing here writes to the database, and nothing here loads a map: gene sets need no drawing material, so this is a matter of seconds rather than the couple of minutes `4_10` spends hydrating.

In [1]:
%store -r

In [2]:
import commute_dm.core
import commute_dm.submaps
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

## Parameters

One entry per pairing, holding exactly what `4_10` holds in its own parameter cell — the collection names, the interface to join over and the levels to walk — plus the output directory of each analysis and mode.

`max_levels` differs between the two for the reason `4_10` gives: the AD influence graph is far denser than an activity-flow map, so a level means much more there. `WITH_SUBUNITS` now means the same thing on both sides — a CellDesigner complex's subunits, and a BEL complex's or composite's members. A BEL **activity** is not a subunit and is always followed: `act(p(X))` is X in another form, the way an active CellDesigner species is still that species.

In [4]:
MIN_N_NODES = 5
WITH_SUBUNITS = True
P_VALUE_CUTOFF = 0.05
MODES = ["upstream", "downstream", "upstream_and_downstream"]

PAIRINGS = [
    {
        "name": "COVID -> PD",
        # The interface is the three-way one, as in `4_10`: a protein has to be
        # shared with the AD KG as well to be looked at here.
        "upstream_collection_name": "COVID_DM_CD_AF",
        "downstream_collection_name": "PD_DM_CD_AF",
        "interface_collection_names": (
            "COVID_DM_CD_AF",
            "PD_DM_CD_AF",
            "AD_KG_CD_AF",
        ),
        "max_levels": [2, 3, 4, 5, 6],
        "goat_dir_paths": {
            "upstream": INTERFACE_PD_GOAT_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_PD_GOAT_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_PD_GOAT_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
        "intersection_dir_paths": {
            "upstream": INTERFACE_PD_INTERSECTION_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_PD_INTERSECTION_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_PD_INTERSECTION_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
    },
    {
        # Two-way, as in `4_10`: the pairing *is* COVID and the AD KG, which
        # `2_10` stores as an ordinary CellDesigner collection.
        "name": "COVID -> AD",
        "upstream_collection_name": "COVID_DM_CD_AF",
        "downstream_collection_name": "AD_KG_CD_AF",
        "interface_collection_names": (
            "COVID_DM_CD_AF",
            "AD_KG_CD_AF",
        ),
        "max_levels": [1, 2, 3],
        "goat_dir_paths": {
            "upstream": INTERFACE_AD_GOAT_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_AD_GOAT_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_AD_GOAT_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
        "intersection_dir_paths": {
            "upstream": INTERFACE_AD_INTERSECTION_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_AD_INTERSECTION_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_AD_INTERSECTION_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
    },
]

## The interface and the influence structure

For each pairing we compute its interface and load the influence structure the walk needs. `load_signed_influences` is a handful of small queries and hydrates no map: these analyses walk node ids and read annotations off the database, so the `source_map` that `4_10` pays a couple of minutes for would buy them nothing.

In [5]:
for pairing in PAIRINGS:
    pairing["interface"] = commute_dm.core.get_interface(
        session, pairing["interface_collection_names"]
    )
    pairing["influences"] = commute_dm.submaps.load_signed_influences(
        session,
        [
            pairing["upstream_collection_name"],
            pairing["downstream_collection_name"],
        ],
    )
    # Once per pairing, not once per mode: this is a query per interface protein.
    pairing["display_names"] = commute_dm.core.get_interface_display_names(
        session, pairing["interface"]
    )

{
    pairing["name"]: {
        "n_interface": len(pairing["interface"]),
    }
    for pairing in PAIRINGS
}

{'COVID -> PD': {'n_interface': 111}, 'COVID -> AD': {'n_interface': 159}}

## GOAT enrichment

For each pairing, each mode and each level, the gene set of the selection around every interface protein is tested against every gene list. Output is one CSV per gene list per level, plus a `summary.csv` listing, per interface protein, the levels at which it came out significant — ordered by how often it did.

The three modes are the three ways to read a comorbidity sub-graph: what the interface protein is downstream of in COVID (`upstream`), what it drives in the comorbid disease (`downstream`), and both at once.

`MIN_N_NODES` applies to **both** directions, so an interface protein is skipped at a level unless each side reaches at least that many nodes — which is why the counts above are much larger than the number of proteins actually tested.

In [6]:
for pairing in PAIRINGS:
    for mode in MODES:
        output_dir_path = pairing["goat_dir_paths"][mode]
        commute_dm.utils.remake_dir(output_dir_path)
        print(f"GOAT: {pairing['name']}, {mode} -> {output_dir_path}")
        commute_dm.core.make_goat_analysis_from_interface(
            session,
            pairing["interface"],
            pairing["influences"],
            GOAT_GENE_LISTS_BUILD_DIR,
            output_dir_path,
            upstream_collection_name=pairing["upstream_collection_name"],
            downstream_collection_name=pairing["downstream_collection_name"],
            mode=mode,
            max_levels=pairing["max_levels"],
            with_subunits=WITH_SUBUNITS,
            p_value_cutoff=P_VALUE_CUTOFF,
            min_n_nodes=MIN_N_NODES,
            display_names=pairing["display_names"],
        )

GOAT: COVID -> PD, upstream -> ../../build/results/interface/analysis/pd/goat_upstream
GOAT: COVID -> PD, downstream -> ../../build/results/interface/analysis/pd/goat_downstream
GOAT: COVID -> PD, upstream_and_downstream -> ../../build/results/interface/analysis/pd/goat_upstream_and_downstream
GOAT: COVID -> AD, upstream -> ../../build/results/interface/analysis/ad/goat_upstream
GOAT: COVID -> AD, downstream -> ../../build/results/interface/analysis/ad/goat_downstream
GOAT: COVID -> AD, upstream_and_downstream -> ../../build/results/interface/analysis/ad/goat_upstream_and_downstream


## Intersection with the significant genes

The same selections, read the naive way: how much of each sub-graph's gene set is among the genes a list marks significant. Gene sets are `hgnc.symbol` here rather than `ncbigene`, because that is what the gene lists' `symbol` column holds.

This is not a test — it has no null model and no p-value, and a big sub-graph will overlap more by construction, which is exactly what GOAT above corrects for. It is here because it says *which* genes, where GOAT only says whether. The `summary.csv` orders interface proteins by the mean `|intersection| / |gene set|` over the levels.

In [7]:
for pairing in PAIRINGS:
    for mode in MODES:
        output_dir_path = pairing["intersection_dir_paths"][mode]
        commute_dm.utils.remake_dir(output_dir_path)
        print(f"intersection: {pairing['name']}, {mode} -> {output_dir_path}")
        commute_dm.core.make_intersection_analysis_from_interface(
            session,
            pairing["interface"],
            pairing["influences"],
            GOAT_GENE_LISTS_BUILD_DIR,
            output_dir_path,
            upstream_collection_name=pairing["upstream_collection_name"],
            downstream_collection_name=pairing["downstream_collection_name"],
            mode=mode,
            max_levels=pairing["max_levels"],
            with_subunits=WITH_SUBUNITS,
            min_n_nodes=MIN_N_NODES,
            display_names=pairing["display_names"],
        )

intersection: COVID -> PD, upstream -> ../../build/results/interface/analysis/pd/intersection_upstream
intersection: COVID -> PD, downstream -> ../../build/results/interface/analysis/pd/intersection_downstream
intersection: COVID -> PD, upstream_and_downstream -> ../../build/results/interface/analysis/pd/intersection_upstream_and_downstream
intersection: COVID -> AD, upstream -> ../../build/results/interface/analysis/ad/intersection_upstream
intersection: COVID -> AD, downstream -> ../../build/results/interface/analysis/ad/intersection_downstream
intersection: COVID -> AD, upstream_and_downstream -> ../../build/results/interface/analysis/ad/intersection_upstream_and_downstream


## What came out

The head of each `summary.csv`: the interface proteins whose sub-graph was most often significant (GOAT) or most heavily overlapping (intersection), per pairing and mode.

Every output table is keyed by UniProt accession (`identifier`) with the HGNC symbol beside it (`display_name`) — the same name `4_10` gives the protein's map directory. The accession stays because it is what joins the collections and what is guaranteed unique: `BBC3` is the symbol of two accessions in the current data, and `get_interface_display_names` disambiguates those as `BBC3_<accession>` rather than letting two proteins collapse into one row.

In [8]:
import os.path

import pandas

for pairing in PAIRINGS:
    for analysis, dir_paths in (
        ("goat", pairing["goat_dir_paths"]),
        ("intersection", pairing["intersection_dir_paths"]),
    ):
        for mode in MODES:
            summary_file_path = os.path.join(dir_paths[mode], "summary.csv")
            if not os.path.exists(summary_file_path):
                continue
            summary_df = pandas.read_csv(summary_file_path, index_col=0)
            print(
                f"\n=== {pairing['name']} / {analysis} / {mode} "
                f"({len(summary_df)} proteins)"
            )
            display(summary_df.head(10))


=== COVID -> PD / goat / upstream (45 proteins)


,identifier,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,deseqobj_COVID19_dopaminergic_GSE174745.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,P99999,CYCS,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5]",...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[2],[],"[2, 3, 4, 5]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
1,Q96P20,NLRP3,"[2, 4, 5, 6]",[],[],"[5, 6]","[2, 4, 5, 6]",[2],[],[],...,"[2, 3, 4, 5, 6]",[6],"[4, 5, 6]",[],"[2, 3]",[],"[3, 4, 5, 6]",[],"[4, 5, 6]",[]
2,P25963,NFKBIA,"[3, 4, 5, 6]",[],"[2, 3, 5, 6]",[],"[4, 5, 6]",[],[],[],...,[],"[2, 3, 4, 5, 6]","[4, 5, 6]",[],[],[],"[4, 5, 6]",[],"[4, 5, 6]",[]
3,P01584,IL1B,"[3, 4, 5, 6]",[],[],[],"[4, 5, 6]",[],[],[],...,"[4, 6]",[],"[4, 5, 6]",[],[],[],"[4, 5, 6]",[],"[4, 5, 6]",[]
4,P35638,DDIT3,"[3, 4, 5, 6]",[],"[3, 5, 6]",[],"[5, 6]",[],[],[],...,[],"[3, 5, 6]","[4, 5, 6]",[],[],[],"[4, 5, 6]",[],"[5, 6]",[]
5,Q16539,MAPK14,"[3, 4, 5, 6]",[],[6],[],"[4, 5, 6]",[],[],[],...,[],[6],"[4, 5, 6]",[],[],[],"[4, 5, 6]",[],"[4, 5, 6]",[]
6,P23560,BDNF,"[4, 5, 6]",[],[],"[5, 6]","[5, 6]",[],[],[],...,[6],[],"[5, 6]","[5, 6]",[],[],"[5, 6]",[],"[5, 6]",[]
7,P01138,NGF,"[5, 6]",[],[],"[5, 6]","[5, 6]",[],[],[],...,[6],[],"[5, 6]","[5, 6]",[],[],"[5, 6]",[],"[5, 6]",[]
8,Q07812,BAX,"[2, 3, 4, 5, 6]",[],[],[],[],[],[],[],...,[],[],"[4, 5, 6]",[],[],[],"[4, 5, 6]",[],[],[]
9,P55957,BID,"[3, 4, 5, 6]",[],[],[],[],[],[],[],...,[],[],"[4, 5, 6]",[],[],[],"[4, 5, 6]",[],[],[]



=== COVID -> PD / goat / downstream (45 proteins)


,identifier,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,deseqobj_COVID19_dopaminergic_GSE174745.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,P99999,CYCS,"[2, 3, 4, 5]","[4, 5, 6]","[4, 5, 6]",[],"[2, 3, 4, 5, 6]","[4, 5, 6]",[],[],...,[4],"[4, 5, 6]",[2],[],"[4, 5, 6]",[],[2],[],"[2, 3, 4, 5, 6]","[4, 5, 6]"
1,O60603,TLR2,"[2, 3, 4, 5, 6]",[2],[],[6],"[2, 3, 4, 5, 6]",[],[],[],...,"[2, 3, 4, 6]",[],"[4, 5, 6]",[6],[],[],"[4, 5, 6]",[],"[2, 3, 4, 5, 6]",[2]
2,Q04206,RELA,"[2, 3, 4, 5, 6]",[2],[],[],"[3, 4, 5, 6]",[],[],[],...,"[2, 3, 4, 5, 6]",[],"[2, 4, 5, 6]",[],[],[],"[2, 4, 5, 6]",[],"[3, 4, 5, 6]","[2, 3]"
3,P01584,IL1B,"[2, 3, 4, 5, 6]",[2],[],[6],"[3, 4, 5, 6]",[],[],[],...,"[3, 6]",[],"[4, 5, 6]",[6],[],[],"[4, 5, 6]",[],"[3, 4, 5, 6]",[2]
4,P01375,TNF,"[2, 3, 4, 5, 6]",[2],[],[6],"[3, 4, 5, 6]",[],[],[],...,[6],[],"[4, 5, 6]",[6],[],[],"[4, 5, 6]",[],"[3, 4, 5, 6]",[2]
5,P19838,NFKB1,"[2, 3, 4, 5, 6]",[2],[],[],"[5, 6]",[],[],[],...,"[3, 4, 5, 6]",[],"[4, 5, 6]",[],[],[],"[4, 5, 6]",[],"[5, 6]","[2, 3, 4]"
6,P10599,TXN,[6],"[5, 6]","[5, 6]",[],"[3, 4, 5, 6]","[5, 6]",[],[],...,[],"[5, 6]",[],[],"[5, 6]",[],[],[],"[3, 4, 5, 6]","[5, 6]"
7,P01019,AGT,"[3, 4, 5, 6]",[],[],[6],"[4, 5, 6]",[],[],[],...,[],[],"[4, 5, 6]",[],[],[],"[4, 5, 6]",[],"[4, 5, 6]",[]
8,Q16539,MAPK14,"[2, 3, 4, 5, 6]",[],[],[],"[3, 4, 5]",[],[],[],...,[6],[],"[4, 5]",[],[],[],"[4, 5]",[],"[4, 5]",[]
9,Q99836,MYD88,"[4, 5, 6]",[],[],[],"[5, 6]",[],[],[],...,"[4, 5, 6]",[],"[4, 5]",[],[],[],"[4, 5]",[],"[5, 6]",[4]



=== COVID -> PD / goat / upstream_and_downstream (45 proteins)


...

,identifier,score,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,P19838,0.450000,NFKB1,"['1: 0/1', '2: 0/1', '3: 0/1']","['1: 1/1', '2: 1/1', '3: 1/1']","['1: 1/1', '2: 1/1', '3: 1/1']","['1: 0/1', '2: 0/1', '3: 0/1']","['1: 1/1', '2: 1/1', '3: 1/1']","['1: 0/1', '2: 0/1', '3: 0/1']","['1: 0/1', '2: 0/1', '3: 0/1']",...,"['1: 1/1', '2: 1/1', '3: 1/1']","['1: 1/1', '2: 1/1', '3: 1/1']","['1: 1/1', '2: 1/1', '3: 1/1']","['1: 0/1', '2: 0/1', '3: 0/1']","['1: 0/1', '2: 0/1', '3: 0/1']","['1: 0/1', '2: 0/1', '3: 0/1']","['1: 1/1', '2: 1/1', '3: 1/1']","['1: 0/1', '2: 0/1', '3: 0/1']","['1: 1/1', '2: 1/1', '3: 1/1']","['1: 1/1', '2: 1/1', '3: 1/1']"
1,Q14116,0.356250,IL18,"['2: 3/4', '3: 3/5']","['2: 1/4', '3: 1/5']","['2: 1/4', '3: 2/5']","['2: 0/4', '3: 0/5']","['2: 4/4', '3: 5/5']","['2: 0/4', '3: 0/5']","['2: 0/4', '3: 0/5']",...,"['2: 1/4', '3: 1/5']","['2: 1/4', '3: 2/5']","['2: 4/4', '3: 5/5']","['2: 0/4', '3: 0/5']","['2: 1/4', '3: 1/5']","['2: 0/4', '3: 0/5']","['2: 4/4', '3: 5/5']","['2: 1/4', '3: 1/5']","['2: 3/4', '3: 4/5']","['2: 1/4', '3: 1/5']"
2,Q12933,0.316667,TRAF2,"['2: 2/6', '3: 2/6']","['2: 4/6', '3: 4/6']","['2: 2/6', '3: 2/6']","['2: 0/6', '3: 0/6']","['2: 4/6', '3: 4/6']","['2: 0/6', '3: 0/6']","['2: 0/6', '3: 0/6']",...,"['2: 0/6', '3: 0/6']","['2: 2/6', '3: 2/6']","['2: 5/6', '3: 5/6']","['2: 0/6', '3: 0/6']","['2: 0/6', '3: 0/6']","['2: 0/6', '3: 0/6']","['2: 5/6', '3: 5/6']","['2: 1/6', '3: 1/6']","['2: 4/6', '3: 4/6']","['2: 5/6', '3: 5/6']"
3,Q00535,0.300000,CDK5,['3: 0/1'],['3: 1/1'],['3: 1/1'],['3: 0/1'],['3: 1/1'],['3: 0/1'],['3: 0/1'],...,['3: 0/1'],['3: 1/1'],['3: 0/1'],['3: 0/1'],['3: 0/1'],['3: 0/1'],['3: 0/1'],['3: 0/1'],['3: 1/1'],['3: 1/1']
4,P35228,0.273125,NOS2,"['2: 3/5', '3: 4/8']","['2: 2/5', '3: 2/8']","['2: 1/5', '3: 1/8']","['2: 0/5', '3: 0/8']","['2: 3/5', '3: 4/8']","['2: 0/5', '3: 0/8']","['2: 0/5', '3: 0/8']",...,"['2: 1/5', '3: 1/8']","['2: 1/5', '3: 1/8']","['2: 3/5', '3: 5/8']","['2: 0/5', '3: 0/8']","['2: 1/5', '3: 1/8']","['2: 0/5', '3: 0/8']","['2: 3/5', '3: 5/8']","['2: 1/5', '3: 2/8']","['2: 3/5', '3: 5/8']","['2: 3/5', '3: 4/8']"
5,P10599,0.272500,TXN,"['2: 0/2', '3: 1/5']","['2: 1/2', '3: 2/5']","['2: 0/2', '3: 2/5']","['2: 0/2', '3: 0/5']","['2: 2/2', '3: 3/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']",...,"['2: 0/2', '3: 0/5']","['2: 1/2', '3: 3/5']","['2: 0/2', '3: 2/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 2/5']","['2: 2/2', '3: 2/5']","['2: 2/2', '3: 3/5']","['2: 1/2', '3: 3/5']"
6,Q99683,0.271429,MAP3K5,"['2: 2/7', '3: 2/7']","['2: 4/7', '3: 4/7']","['2: 2/7', '3: 2/7']","['2: 0/7', '3: 0/7']","['2: 4/7', '3: 4/7']","['2: 0/7', '3: 0/7']","['2: 0/7', '3: 0/7']",...,"['2: 0/7', '3: 0/7']","['2: 2/7', '3: 2/7']","['2: 5/7', '3: 5/7']","['2: 0/7', '3: 0/7']","['2: 0/7', '3: 0/7']","['2: 0/7', '3: 0/7']","['2: 5/7', '3: 5/7']","['2: 1/7', '3: 1/7']","['2: 4/7', '3: 4/7']","['2: 5/7', '3: 5/7']"
7,P49662,0.265000,CASP4,"['2: 1/2', '3: 1/5']","['2: 1/2', '3: 2/5']","['2: 1/2', '3: 2/5']","['2: 0/2', '3: 0/5']","['2: 1/2', '3: 2/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']",...,"['2: 0/2', '3: 0/5']","['2: 1/2', '3: 2/5']","['2: 1/2', '3: 4/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 1/2', '3: 3/5']","['2: 1/2', '3: 1/5']","['2: 1/2', '3: 2/5']","['2: 1/2', '3: 2/5']"
8,P17861,0.261667,XBP1,"['2: 3/6', '3: 4/10']","['2: 3/6', '3: 5/10']","['2: 1/6', '3: 2/10


=== COVID -> AD / intersection / downstream (51 proteins)


,identifier,score,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,P01100,0.267500,FOS,"['2: 2/2', '3: 2/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 2/2', '3: 4/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']",...,"['2: 0/2', '3: 0/5']","['2: 1/2', '3: 1/5']","['2: 2/2', '3: 3/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 1/5']","['2: 0/2', '3: 0/5']","['2: 2/2', '3: 4/5']","['2: 0/2', '3: 0/5']","['2: 2/2', '3: 4/5']","['2: 0/2', '3: 0/5']"
1,Q14116,0.250000,IL18,"['2: 1/1', '3: 1/2']","['2: 0/1', '3: 1/2']","['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']","['2: 1/1', '3: 1/2']","['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']",...,"['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']","['2: 1/1', '3: 1/2']","['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']","['2: 1/1', '3: 1/2']","['2: 0/1', '3: 0/2']","['2: 1/1', '3: 1/2']","['2: 0/1', '3: 1/2']"
2,P04637,0.231667,TP53,"['2: 2/3', '3: 2/5']","['2: 1/3', '3: 2/5']","['2: 1/3', '3: 1/5']","['2: 0/3', '3: 0/5']","['2: 1/3', '3: 3/5']","['2: 0/3', '3: 0/5']","['2: 0/3', '3: 0/5']",...,"['2: 0/3', '3: 0/5']","['2: 1/3', '3: 1/5']","['2: 2/3', '3: 2/5']","['2: 0/3', '3: 0/5']","['2: 0/3', '3: 0/5']","['2: 0/3', '3: 0/5']","['2: 1/3', '3: 1/5']","['2: 1/3', '3: 2/5']","['2: 1/3', '3: 3/5']","['2: 1/3', '3: 3/5']"
3,Q12933,0.208333,TRAF2,"['2: 1/2', '3: 1/3']","['2: 1/2', '3: 1/3']","['2: 0/2', '3: 0/3']","['2: 0/2', '3: 0/3']","['2: 1/2', '3: 1/3']","['2: 0/2', '3: 0/3']","['2: 0/2', '3: 0/3']",...,"['2: 0/2', '3: 0/3']","['2: 0/2', '3: 0/3']","['2: 1/2', '3: 1/3']","['2: 0/2', '3: 0/3']","['2: 0/2', '3: 0/3']","['2: 0/2', '3: 0/3']","['2: 1/2', '3: 1/3']","['2: 1/2', '3: 1/3']","['2: 1/2', '3: 1/3']","['2: 1/2', '3: 1/3']"
4,Q07812,0.200000,BAX,"['2: 0/2', '3: 0/2']","['2: 1/2', '3: 1/2']","['2: 1/2', '3: 1/2']","['2: 0/2', '3: 0/2']","['2: 1/2', '3: 1/2']","['2: 0/2', '3: 0/2']","['2: 0/2', '3: 0/2']",...,"['2: 0/2', '3: 0/2']","['2: 1/2', '3: 1/2']","['2: 1/2', '3: 1/2']","['2: 0/2', '3: 0/2']","['2: 0/2', '3: 0/2']","['2: 0/2', '3: 0/2']","['2: 1/2', '3: 1/2']","['2: 0/2', '3: 0/2']","['2: 1/2', '3: 1/2']","['2: 1/2', '3: 1/2']"
5,Q04206,0.192795,RELA,"['1: 1/2', '2: 14/54', '3: 48/223']","['1: 0/2', '2: 11/54', '3: 50/223']","['1: 0/2', '2: 5/54', '3: 36/223']","['1: 0/2', '2: 0/54', '3: 0/223']","['1: 1/2', '2: 19/54', '3: 78/223']","['1: 0/2', '2: 0/54', '3: 0/223']","['1: 0/2', '2: 0/54', '3: 1/223']",...,"['1: 0/2', '2: 1/54', '3: 2/223']","['1: 0/2', '2: 5/54', '3: 34/223']","['1: 1/2', '2: 24/54', '3: 83/223']","['1: 0/2', '2: 0/54', '3: 0/223']","['1: 1/2', '2: 2/54', '3: 10/223']","['1: 0/2', '2: 0/54', '3: 0/223']","['1: 1/2', '2: 25/54', '3: 90/223']","['1: 1/2', '2: 10/54', '3: 36/223']","['1: 1/2', '2: 20/54', '3: 77/223']","['1: 0/2', '2: 17/54', '3: 63/223']"
6,P36222,0.191728,CHI3L1,"['1: 3/7', '2: 8/25', '3: 42/178']","['1: 3/7', '2: 8/25', '3: 38/178']","['1: 3/7', '2: 5/25', '3: 28/178']","['1: 0/7', '2: 0/25', '3: 1/178']","['1: 3/7', '2: 7/25', '3: 65/178']","['1: 0/7', '2: 0/25', '3: 0/178']","['1: 0/7', '2: 0/25', '3: 1/178']",...,"['1: 2/7', '2: 4/25', '3: 5/178']","['1: 3/7', '2: 5/25', '3: 28/178']","['1: 4/7', '2: 8/25', '3: 72/178']","['1: 0/7', '2: 0/25', '3: 0/178']","['1: 0/7', '2: 0/25', '3: 8/178']","['1: 0/7', '2: 0/25', '3: 0/178']","['1: 4/7', '2: 8/25', '3: 77/178']","['1: 0/7', '2: 1/25', '3: 25/178']","['1: 3/7', '2: 7/25', '3: 65/178']","['1: 3/7', '2: 10/25', '3: 48/178']"
7,Q165


=== COVID -> AD / intersection / upstream_and_downstream (51 proteins)


,identifier,score,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,Q14116,0.335417,IL18,"['2: 3/4', '3: 3/6']","['2: 1/4', '3: 2/6']","['2: 1/4', '3: 2/6']","['2: 0/4', '3: 0/6']","['2: 4/4', '3: 5/6']","['2: 0/4', '3: 0/6']","['2: 0/4', '3: 0/6']",...,"['2: 1/4', '3: 1/6']","['2: 1/4', '3: 2/6']","['2: 4/4', '3: 5/6']","['2: 0/4', '3: 0/6']","['2: 1/4', '3: 1/6']","['2: 0/4', '3: 0/6']","['2: 4/4', '3: 5/6']","['2: 1/4', '3: 1/6']","['2: 3/4', '3: 4/6']","['2: 1/4', '3: 2/6']"
1,Q12933,0.254464,TRAF2,"['2: 2/7', '3: 2/8']","['2: 4/7', '3: 4/8']","['2: 2/7', '3: 2/8']","['2: 0/7', '3: 0/8']","['2: 4/7', '3: 4/8']","['2: 0/7', '3: 0/8']","['2: 0/7', '3: 0/8']",...,"['2: 0/7', '3: 0/8']","['2: 2/7', '3: 2/8']","['2: 5/7', '3: 5/8']","['2: 0/7', '3: 0/8']","['2: 0/7', '3: 0/8']","['2: 0/7', '3: 0/8']","['2: 5/7', '3: 5/8']","['2: 1/7', '3: 1/8']","['2: 4/7', '3: 4/8']","['2: 5/7', '3: 5/8']"
2,P04637,0.248438,TP53,"['2: 2/8', '3: 2/16']","['2: 1/8', '3: 6/16']","['2: 3/8', '3: 4/16']","['2: 0/8', '3: 0/16']","['2: 5/8', '3: 10/16']","['2: 0/8', '3: 0/16']","['2: 0/8', '3: 0/16']",...,"['2: 0/8', '3: 0/16']","['2: 3/8', '3: 4/16']","['2: 7/8', '3: 8/16']","['2: 0/8', '3: 0/16']","['2: 0/8', '3: 0/16']","['2: 0/8', '3: 0/16']","['2: 6/8', '3: 7/16']","['2: 3/8', '3: 6/16']","['2: 5/8', '3: 11/16']","['2: 1/8', '3: 7/16']"
3,P01100,0.234524,FOS,"['2: 3/6', '3: 5/14']","['2: 1/6', '3: 5/14']","['2: 1/6', '3: 3/14']","['2: 0/6', '3: 0/14']","['2: 3/6', '3: 7/14']","['2: 0/6', '3: 0/14']","['2: 0/6', '3: 0/14']",...,"['2: 0/6', '3: 0/14']","['2: 2/6', '3: 4/14']","['2: 4/6', '3: 9/14']","['2: 0/6', '3: 0/14']","['2: 0/6', '3: 1/14']","['2: 0/6', '3: 0/14']","['2: 4/6', '3: 10/14']","['2: 1/6', '3: 2/14']","['2: 4/6', '3: 8/14']","['2: 1/6', '3: 5/14']"
4,P25963,0.231897,NFKBIA,"['2: 3/16', '3: 9/29']","['2: 6/16', '3: 11/29']","['2: 4/16', '3: 6/29']","['2: 0/16', '3: 0/29']","['2: 10/16', '3: 11/29']","['2: 0/16', '3: 0/29']","['2: 0/16', '3: 0/29']",...,"['2: 1/16', '3: 1/29']","['2: 3/16', '3: 5/29']","['2: 10/16', '3: 16/29']","['2: 0/16', '3: 0/29']","['2: 1/16', '3: 1/29']","['2: 0/16', '3: 0/29']","['2: 10/16', '3: 17/29']","['2: 5/16', '3: 6/29']","['2: 11/16', '3: 12/29']","['2: 6/16', '3: 12/29']"
5,Q99683,0.226442,MAP3K5,"['2: 2/8', '3: 3/13']","['2: 4/8', '3: 6/13']","['2: 2/8', '3: 2/13']","['2: 0/8', '3: 0/13']","['2: 4/8', '3: 6/13']","['2: 0/8', '3: 0/13']","['2: 0/8', '3: 0/13']",...,"['2: 0/8', '3: 0/13']","['2: 2/8', '3: 2/13']","['2: 5/8', '3: 8/13']","['2: 0/8', '3: 0/13']","['2: 0/8', '3: 0/13']","['2: 0/8', '3: 0/13']","['2: 5/8', '3: 8/13']","['2: 1/8', '3: 2/13']","['2: 4/8', '3: 5/13']","['2: 5/8', '3: 7/13']"
6,O43521,0.216667,BCL2L11,['3: 1/9'],['3: 4/9'],['3: 3/9'],['3: 0/9'],['3: 4/9'],['3: 0/9'],['3: 0/9'],...,['3: 0/9'],['3: 3/9'],['3: 5/9'],['3: 0/9'],['3: 0/9'],['3: 0/9'],['3: 5/9'],['3: 2/9'],['3: 5/9'],['3: 4/9']
7,P99999,0.205204,CYCS,"['1: 1/19', '2: 13/144', '3: 30/234']","['1: 8/19', '2: 85/144', '3: 109/234']","['1: 5/19', '2: 34/144', '3: 43/234']","['1: 0/19', '2: 0/144', '3: 0/234']","['1: 11/19', '2: 73/144', '3: 97/234']","['1: 0/19', '2: 0/144', '3: 0/234']","['1: 1/19', '2: 3/144', '3: 3/234']",...,"['1: 0/19', '2: 4/144', '3: 6/234']","['1: 6/19', '2: 43/144', '3: 51/234']","['1: 6/19', '2: 53/144', '3: 89/234']","['1: 0/19', '2: 0/144', '3: 0/234']","['1: 0/19', '2: 13/144', '3: 14/234']","['1: 0/19', '2: 0/144', '3: 0/234']","['1: 7/19', '2: 56/144', '3: 94/234']","